# Univariate Numerical EDA

## Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from matplotlib.ticker import MultipleLocator

## Data Preparation

In [ ]:
data_df = pd.read_csv("../data/sample.csv")
feature_analysis_df = pd.read_csv("../data/column_nature.sample.csv")
data_df.head(3)

## Helper Functions

### Separator

In [ ]:
def separator(text: str = ""):
    if text != "":
        text = " " + text + " "

    text_to_print = (
        "=============="
        + text
        + "============================================================================================"
    )

    if text != "":
        text_to_print = text_to_print[: -len(text)]
    print(text_to_print)

### Sampling Helper

In [ ]:
def sampling_helper(df: pd.DataFrame, sampling: int = 0):
    if sampling != 0:
        target_df = df.sample(sampling)
    else:
        target_df = df
    return target_df

### Central Tendency

In [ ]:
def central_tendency(df: pd.DataFrame, column_name: str, sampling: int = 0):
    if column_name in df.columns.to_list():
        print("Central Tendency of Feature: ", column_name)

        target_df = sampling_helper(df, sampling)

        mean = target_df[column_name].mean()
        median = target_df[column_name].median()
        mode = list(target_df[column_name].mode())

        res = pd.DataFrame(
            {"Central Tendency": ["Mean", "Median", "Mode"], "Values": [mean, median, mode]}
        )

        display(res)


central_tendency(data_df, "DurationHospStayDay")

### Spread

In [ ]:
def spread(df: pd.DataFrame, column_name: str, sampling: int = 0):
    if column_name in df.columns.to_list():
    
        print("Spread of Feature: ", column_name)

        target_df = sampling_helper(df, sampling)

        variance = target_df[column_name].var()
        # annova = 0
        standard_deviation = target_df[column_name].std()
        quartiles = target_df[column_name].quantile([0.25, 0.75])
        interquartile_range = quartiles[0.75] - quartiles[0.25]
        upper_outlier = quartiles[0.75] + 1.5 * interquartile_range
        lower_outlier = quartiles[0.25] - 1.5 * interquartile_range
        outliers = list(
            filter(
                lambda x: x < lower_outlier or x > upper_outlier,
                target_df[column_name].tolist(),
            )
        )

        res = pd.DataFrame(
            {
                "Spread": [
                    "variance",
                    # "annova",
                    "standard deviation",
                    "interquartile range",
                    "outliers",
                ],
                "Values": [
                    variance,
                    #    annova,
                    standard_deviation,
                    interquartile_range,
                    outliers,
                ],
            }
        )

        print("outliers >>: ", outliers)
        return res


spread(data_df, "DurationHospStayDay")

### Skewness and Kurtosis

In [ ]:
def skewness_and_kurtosis(df: pd.DataFrame, column_name: str, sampling: int = 0):
    if column_name in df.columns.to_list():
    
        print("Skewness and Kurtosis of Feature: ", column_name)

        target_df = sampling_helper(df, sampling)

        skewness = target_df[column_name].skew()
        kurtosis = target_df[column_name].kurtosis()

        res = pd.DataFrame(
            {
                "Skewness and Kurtosis": ["Skewness", "Kurtosis"],
                "Values": [skewness, kurtosis],
            }
        )

        display(res)


skewness_and_kurtosis(data_df, "DurationHospStayDay")

### Visualization

In [ ]:
def graphical_eda(df: pd.DataFrame, column_name: str):
    if column_name in df.columns.to_list():
    
        print()
        print(
            "=============================================================================="
        )
        print(f"EDA for [{column_name}]")

        ret = df[column_name].plot.box(title="Box Plot", return_type="both")
        ret.ax.get_figure().savefig("./outliers/" + column_name)
        plt.show()
        whiskers_bound = [item.get_ydata()[1] for item in ret.lines['whiskers']]
        print("lower bound:{}, upper bound:{}".format(whiskers_bound[0], whiskers_bound[1]))

        df[column_name].value_counts().plot.hist(title="Frequency of values")
        plt.show()

        sm.qqplot(df[column_name], line="45")
        plt.show()

## EDA

In [ ]:
categorical_features_df = feature_analysis_df.loc[
    feature_analysis_df["columnNature"] == "numerical"
]

outlier_dict = {}

for ix, row in categorical_features_df.iterrows():
    column_name = row["columnName"]

    separator("EDA for Numerical column: " + column_name)
    central_tendency(data_df, column_name)
    spread_df = spread(data_df, column_name)
    display(spread_df)
    skewness_and_kurtosis(data_df, column_name)
    graphical_eda(data_df, column_name)

    if column_name in data_df.columns.to_list():
        outlier_dict[column_name] = spread_df["Values"].loc[3]

    print()

## LOS Histogram

In [ ]:
import numpy as np
subplot = data_df["DurationHospStayDay"].hist(figsize=[13,10])
x_ticks = np.arange(0, 300, 10)
subplot.set_xticks(x_ticks)
plt.xlabel("Length of Stay")
plt.ylabel("Count")
plt.title("Length of Stay Histogram")
plt.show()

In [ ]:
plt.rc('font', size=14)

# Create a figure with two subplots sharing the same x-axis
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 10), sharex=True, gridspec_kw={'height_ratios': [4, 1]})

# Plot the histogram on the first subplot
ax1.hist(data_df["DurationHospStayDay"], bins=30, edgecolor='black', color='white')
ax1.set_ylabel("Frequency")
# DurationHospStayDay Histogram with Boxplot
ax1.set_title("Duration of Hospital Stay")

# Plot the boxplot on the second subplot
box_plot = ax2.boxplot(data_df["DurationHospStayDay"], vert=False)
for median in box_plot['medians']:
    median.set_color('black')
ax2.set_xlabel("Length of Stay")
ax2.set_yticks([])  # Hide the y-axis ticks for the boxplot

# Set x-ticks
x_ticks = np.arange(0, 300, 10)
ax1.set_xticks(x_ticks)
ax1.xaxis.set_major_locator(MultipleLocator(50))
ax1.xaxis.set_minor_locator(MultipleLocator(10))

# Annotate the median and whiskers upper bound value on the boxplot
median_value = data_df["DurationHospStayDay"].median()
whiskers_upper_bound = box_plot['whiskers'][1].get_ydata()[1]
whiskers_upper_bound = 93.0
ax2.annotate(median_value, 
             xy=(median_value, 1), 
             xytext=(median_value, 1.35),
             arrowprops=dict(arrowstyle='->', color='black'), 
             fontsize=12, 
             color='black')
ax2.annotate(whiskers_upper_bound, 
             xy=(whiskers_upper_bound, 1), 
             xytext=(whiskers_upper_bound, 1.35),
             arrowprops=dict(arrowstyle='->', color='black'), 
             fontsize=12, 
             color='black')

# Show the plot
plt.savefig("DurationHospStayDay_histogram_boxplot.png", dpi=300)
plt.show()

## Save Outliers into csv

In [ ]:
outliers_df = pd.DataFrame(data={"columnName": outlier_dict.keys(), "outliers": outlier_dict.values()})
outliers_df

In [ ]:
outliers_df.to_csv("outliers_numerical_columns.csv", index=False)

## Alive only EDA

In [ ]:
alive_data = data_df[data_df['Outcome'] != 'Dead'].copy()
alive_data.shape

In [ ]:
for ix, row in categorical_features_df.iterrows():
    column_name = row["columnName"]

    separator("EDA for Numerical column: " + column_name)
    central_tendency(alive_data, column_name)
    spread_df = spread(alive_data, column_name)
    display(spread_df)
    print()

## Dead Only EDA

In [ ]:
dead_data = data_df[data_df['Outcome'] == 'Dead'].copy()
dead_data.shape

In [ ]:
for ix, row in categorical_features_df.iterrows():
    column_name = row["columnName"]

    separator("EDA for Numerical column: " + column_name)
    central_tendency(dead_data, column_name)
    spread_df = spread(dead_data, column_name)
    display(spread_df)
    print()